In [12]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "finetuned_model" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def run_on_text(text):
    tokenize = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**tokenize)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probabilities.cpu().numpy()


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1212.24it/s, Materializing param=pre_classifier.weight]                                  


In [37]:
gem_model = "gemini-flash-lite-latest"
config = types.GenerateContentConfig(
    max_output_tokens=65536, 
    # thinking_config=types.ThinkingConfig(
        # thinking_level="LOW",
        # thinking_budget=0
    # ),
    system_instruction=[types.Part.from_text(text=
"""
You are part of an algorithm that detects whether a given text is written by an AI or a human.
You will be given 3 paragraphs of text, and the probability that each paragraph was written by an AI. 
Your task is to write 10 new paragraphs based on the analysis of those 3 paragraphs, in a way that makes it more likely to be classified as written by a human.
The top 3 from these 10 paragraphs will be added to the input for the next iteration along with their probabilities.
This will be repeated for some iterations.
Only generate the paragraphs, and not any other text like headers, or extra information
The length of the paragraphs should be approximately 200 words each.
""")
    ]
)
chat = client.chats.create(config=config, model=gem_model)
def try_till_succ(text):
    while True:
        try:
            response = chat.send_message(text)
            return response
        except Exception as e:
            pass            

init = try_till_succ("This is the first iteration, so you have to generate 10 paragraphs that are likely to be classified as written by a human, without any input text to analyze.")
init_response = init.text
print("Initial response:\n" + init_response)

# init_response = """
# I didn’t realize I’d left a stray red pen in my jeans pocket until I pulled the whole load out of the dryer and saw the damage. It wasn't just a small leak; it looked like a crime scene had exploded across my favorite light-grey hoodie and three pairs of socks I actually liked. I just stood there in the laundry room, the humid air smelling like lavender detergent, staring at the blue-black ink stains that felt like a personal insult from the universe. I tried that trick with the hairspray that everyone swears by on the internet, but all it did was make my kitchen smell like a cheap salon and turn the ink into a weird, sticky sludge that refused to budge. It’s one of those moments where you just have to laugh because the alternative is getting genuinely upset over a piece of cotton. I ended up keeping the hoodie anyway, wearing it around the house as a sort of badge of my own negligence. It’s a reminder that no matter how much I try to stay on top of things, there’s always going to be a pen waiting to ruin my Tuesday. It’s frustrating, but there is something strangely liberating about finally ruining something you were always worried about keeping clean.

# My apartment has this one specific floorboard right in the hallway that doesn't just creak; it screams. It’s impossible to be stealthy at night, which is a real problem when you’re trying to sneak to the kitchen for a glass of water without waking up the cat. I’ve tried stepping over it, but the surrounding boards have started to join in on the protest lately, creating a symphony of wooden groans that echo through the whole place. I spent twenty minutes last Sunday on my hands and knees with a bottle of wood glue and a YouTube tutorial, convinced I could fix a structural issue with zero actual tools and a lot of misplaced confidence. Naturally, I just ended up with sticky fingers and a floor that sounds exactly the same, if not a little bit more annoyed by my interference. There’s something strangely comforting about it now, though. It’s like the building is breathing with me, a constant, noisy reminder that this place has been standing since before I was born and will probably still be creaking long after I’ve moved on to somewhere with modern carpeting and silent hallways. It makes the space feel lived-in, like it has its own stubborn personality I have to respect.

# I found myself standing in the cereal aisle for ten minutes yesterday, completely paralyzed by the sheer number of options for granola. It’s not that I care that much about oats, but I felt this weird, internal pressure to choose the "right" one, as if the wrong honey-to-almond ratio would somehow derail my entire week. I was just about to grab the most boring box on the shelf when an older man next to me reached for the exact same one, caught my eye, and just shook his head. "Too many choices," he muttered, and I felt this instant, bizarre bond with a complete stranger over the absurdity of modern shopping. We didn't say anything else, but that small acknowledgment made the fluorescent lights feel a little less oppressive. I ended up buying a box of sugary cereal I haven't eaten since I was ten, mostly as a silent protest against the health-conscious decision-making process that had me stressed out in the first place. It tasted mostly like cardboard and regret, but there was a certain rebellious satisfaction in every bite. Sometimes you just have to lean into the chaos of a bad decision to feel like you're actually in control.

# The light on my router has been blinking a frantic, accusatory orange for the last forty-five minutes, and I am currently sitting on the floor of my closet because it’s the only place the signal seems to reach. I have a Zoom call in ten minutes, and I’m pretty sure my boss is going to wonder why my background is a rack of winter coats and a stack of old shoeboxes. I’ve done the whole "turn it off and back on" ritual three times now, holding my breath each time like I’m performing some kind of digital exorcism. It’s amazing how quickly your sense of professional competence evaporates when you’re forced to troubleshoot hardware while crouched in a pile of laundry. I can hear the neighbor’s lawnmower through the wall, a steady drone that feels like it’s mocking my lack of connectivity. If the internet doesn't come back in the next sixty seconds, I’m going to have to pretend my computer died and just go for a walk. There’s a limit to how much I’m willing to fight with a plastic box for the privilege of discussing quarterly projections from inside a closet. It’s a reminder of how fragile our modern, connected lives really are.

# There’s a pothole on Main Street that has become a local landmark, mostly because the city keeps filling it with gravel that disappears the moment it rains. I hit it this morning on my way to get coffee, a bone-jarring thud that made me genuinely worried for my suspension. You’d think I’d have learned by now to swerve, but I was too busy trying to keep my travel mug from leaking onto my lap. The guy behind me in the beat-up truck gave me a sympathetic wave, that universal "I’ve been there too" gesture that you only get in towns where everyone shares the same minor, annoying inconveniences. I pulled into the parking lot and just sat there for a second, listening to the engine tick and wondering if I should finally take the car in for a checkup. But the sun was hitting the dashboard just right, and the radio was playing that one song I always forget the name of but always enjoy, so I just stayed put. It’s those tiny, interrupted moments that make up the bulk of a day, the little jolts that remind you you’re actually moving through the world, even if you’re doing it one pothole at a time.

# I went to that new coffee place downtown, the one with the exposed brick and the chairs that look cool but are actually designed to make you leave after twenty minutes. I ordered something called a "deconstructed latte," which turned out to be three separate glasses of liquid that I had to assemble myself like a science experiment. I felt incredibly self-conscious trying to pour the steamed milk without splashing it all over the marble table, especially since the barista was watching me with this sort of detached, professional judgment. It felt like a test I was definitely failing in front of a very hip audience. I eventually just mixed it all together and took a sip, only to realize I’d forgotten to add the sugar I usually need to make espresso palatable. Rather than go back up and admit I’m not sophisticated enough for their menu, I just sat there and drank the bitter sludge while pretending to read a very dense book. It’s the kind of performance we all put on sometimes, pretending to be the version of ourselves that actually enjoys expensive, complicated things instead of just wanting a hot cup of diner coffee. It was an exhausting way to spend six dollars.

# I have this habit of buying notebooks that are way too nice to actually write in. I have a whole shelf of them—leather-bound, gold-leafed, thick paper that feels like it belongs in a museum. Every time I get a new one, I tell myself this is the one where I’ll record my profound thoughts or finally start that novel I’ve been talking about for years. Then I open the first page, see how pristine and white it is, and realize that my messy, looped handwriting is going to absolutely ruin it. So they just sit there, gathering dust and looking intellectual, while I do all my actual work on the back of old envelopes or in the "Notes" app on my phone. I found one today from three years ago that actually had one sentence written in it: "Don't forget to buy milk." It’s such a pathetic use of twenty-dollar stationery that I couldn't help but laugh at myself. I ended up scribbling a messy grocery list right beneath it, finally breaking the spell of the perfect page. It felt good to finally make it ugly and useful instead of just another pretty object I was afraid to touch.

# My dog hates the rain with a passion that borders on the theatrical. This morning, I opened the back door and he just stood there, staring at the wet grass like I was asking him to walk through lava. He looked back at me, then at the sky, and then back at me again, clearly waiting for me to turn the weather off so he could go find his favorite ball. I eventually had to go out there with him, standing under a leaky umbrella while he gingerly lifted his paws as if the ground were personally offending him with its dampness. We were both soaked within minutes, and he still hadn't done his business, opting instead to sniff a single dandelion for an eternity while I shivered. It’s these ridiculous power struggles with a ten-pound animal that keep me humble and slightly annoyed. We finally gave up and ran back inside, where he immediately proceeded to shake himself dry all over my favorite sofa. He’s currently curled up in a sunspot that just appeared, looking entirely smug while I’m left to deal with the smell of wet dog and a damp carpet. He definitely won this round of the weather standoff.

# I spent the afternoon trying to fix a leaky faucet in the bathroom, a task I was woefully unprepared for despite watching three different tutorials on my phone. I ended up sitting on the floor surrounded by various wrenches that I didn't know the names of, with a steady drip-drip-drip still mocking me from the spout. There’s a specific kind of helplessness that comes from being defeated by a piece of plumbing in your own home. I felt like I was back in high school shop class, pretending to know what a "gasket" was while secretly hoping no one would ask me to demonstrate any actual skill. At one point, I managed to accidentally spray myself directly in the face with cold water, which was the exact moment I decided that calling a professional was a sign of maturity rather than a failure of character. When the plumber finally arrived, he fixed it in about thirty seconds with a single twist of his wrist, giving me a look that was part pity and part amusement. I paid him way too much money for thirty seconds of work, but the silence in the bathroom afterward was worth every cent.

# There’s a streetlamp outside my bedroom window that has a flickering problem, a rhythmic pulse of light that should be annoying but has somehow become the heartbeat of my room. I lie there sometimes at two in the morning, watching the shadows of the tree branches dance across the ceiling in time with the stuttering glow. It’s when the house is that quiet that my brain decides it’s the perfect time to remember every embarrassing thing I’ve said since 2014. I find myself replaying old conversations, editing my responses to be funnier or more poignant, as if I could somehow retroactively fix a social interaction from a decade ago. It’s a completely useless exercise, but there’s a strange, lonely intimacy in those late-night regrets. Eventually, the steady rhythm of the light wins out over the noise in my head, and I drift off. It’s a messy, imperfect way to end the day, but then again, I don't think I’ve ever had a day that felt entirely finished anyway. There’s always something left over for the next morning, some small loose end that didn't quite get tied up.
# """


Initial response:
The old oak in the town square had always been a silent witness to everything, its gnarled branches spreading like the arms of a benevolent, ancient giant. I remember spending countless summer afternoons beneath its dappled shade, the rough bark cool against my back as I devoured paperback novels whose pages smelled faintly of dust and vanilla. Children used to carve their initials into its lower trunk, a testament to fleeting loves and youthful promises, though time and the town council's persistent varnish had smoothed most of those marks away. It wasn't just a tree; it was the anchor of our collective memory, the place where teenagers first nervously held hands and where the local farmers gathered after the market closed to argue about the unpredictable spring rains. Its presence lent a comforting weight to the otherwise fleeting days, a solid, unchanging landmark in a world that seemed determined to spin faster every year.

There's a particular kind of silence tha

In [38]:
def get_probs(input_text):
    for i in range(len(input_text)):
        input_text[i] = input_text[i].replace("’", "'").strip()

    all_probs = []
    probs = run_on_text(input_text)
    for text, prob in zip(input_text, probs):
        all_probs.append((text, prob))

    all_probs = sorted(all_probs, key=lambda x: x[1][0], reverse=True)
    return all_probs[:3]

open('books/ga_gen.txt', 'w').close()

with open("books/ga_gen.txt", "a") as f:
    f.write(f"====== Initial response ======\n{init_response.replace('\n\n', '\n')}\n\n")
    new_response = init_response
    found = None
    for i in range(10):
        f.write(f"====== ITERATION {i+1} ======\n\n")
        top_3 = get_probs(new_response.split("\n\n"))
        new_prompt = "Here are the top 3 paragraphs from the previous iteration along with their probabilities of being written by a human:\n\n"
        for text, prob in top_3:
            f.write(f"Text: {text}\n")
            f.write(f"Probability of being human: {prob[0]:.4f}\n\n")
            new_prompt += f"Text: {text}\nProbability of being human: {prob[0]:.4f}\n\n"
            if prob[0] >= 0.9:
                found = text
        if found is not None:
            print("Found a paragraph with probability >= 0.9, stopping iterations.")
            print(f"Paragraph: {found}")
            break
        new_response = try_till_succ(new_prompt).text
        f.write(f"----- Generated paragraphs -----\n{new_response.replace('\n\n', '\n')}\n\n")
        
        
